# Day 04 — File I/O, JSON & CSV
**Module 01: Python + Async + FastAPI for LLM Engineering**  
Vidya Sankalp | Applied GenAI Engineering

**Goal:** Learn to read and write files, understand JSON, and process simple CSV data.

## Learning path
1. Understand what JSON is
2. Convert between Python data and JSON strings (`loads`, `dumps`)
3. Save and read JSON files (`dump`, `load`)
4. Read and write text files safely with `with open()`
5. Read CSV rows with `csv.DictReader`
6. Combine the skills in a small AI-style workflow

**Connection to Day 03:** We'll reuse dictionaries, lists, and functions. Start with the small examples and move to the full workflow only after the basics are clear.

---


## 1 — Imports


In [ ]:
import csv
import io  # Used in the optional CSV examples later in the notebook
# Tools we'll use in this lesson
import json

---


## 2 — First, understand JSON

JSON is a **text format for storing and exchanging structured data**. It often looks like a Python dictionary or list, but JSON is text with its own syntax.

Example JSON data:

```json
{
    "name": "Ravi",
    "age": 30,
    "city": "Hyderabad",
    "is_active": true
}
```

A few important differences from Python:

| JSON | Python |
|---|---|
| Object `{...}` | Dictionary |
| Array `[...]` | List |
| `true` / `false` | `True` / `False` |
| `null` | `None` |
| Strings use double quotes | Python strings may use single or double quotes |

**Remember:** a Python dictionary, a JSON string, and a JSON file are three different things. The four functions below help us move between them.

### The four JSON functions

| Function | Input | Result | Use it when... |
|---|---|---|---|
| `json.dumps(data)` | Python object | JSON string | You need JSON text in memory |
| `json.loads(text)` | JSON string | Python object | You need to read JSON text, such as an API response |
| `json.dump(data, file)` | Python object + open file | Writes JSON into a file | You want to save data to a `.json` file |
| `json.load(file)` | Open JSON file | Python object | You want to read data from a `.json` file |

**Memory trick:** the `s` stands for **string**. `loads()` and `dumps()` work with strings; `load()` and `dump()` work with file objects.

We'll learn the string versions first, then the file versions.

### A. `json.loads()` — JSON string → Python object

Use `loads()` when JSON is already available as text, for example in an API or LLM response.

In [ ]:
# ── json.loads() — string → dict ───────────────────────────
raw = '{"category": "TRACK_ORDER", "confidence": "high"}'

print('Before:', type(raw), raw)
parsed = json.loads(raw)
print('After :', type(parsed), parsed)
print('Access:', parsed['category'])


### B. `json.dumps()` — Python object → JSON string

Use `dumps()` when you need to turn a Python dictionary or list into JSON text.

In [ ]:
# ── json.dumps() — dict → string ───────────────────────────
config = {'model': 'gpt-4o', 'max_tokens': 1024, 'temperature': 0.2}

compact = json.dumps(config)           # single line — for logs
pretty  = json.dumps(config, indent=2) # indented — for .json files

print('Compact:', compact)
print('Type   :', type(compact))   # str
print()
print('Pretty:')
print(pretty)


### C. `json.dump()` — Python object → JSON file

Use `dump()` to write Python data directly into an open file. `indent=4` makes the file easier to read.

In [ ]:
# json.dump  → write a dict directly to an open file
with open('model_config.json', 'w', encoding='utf-8') as f:
    json.dump(config, f, indent=2)

### D. `json.load()` — JSON file → Python object

Use `load()` to read JSON directly from an open file.

In [ ]:
# ── json.load() and json.dump() — file versions ─────────────

# json.load  → read a dict directly from an open file

with open('model_config.json', 'r', encoding='utf-8') as f:
    loaded = json.load(f)

print('Loaded back:', loaded)
print('Type       :', type(loaded).__name__)   # dict


---


## 3 — The `with` Statement — Safe File Handling

The `with` statement guarantees the file is **closed automatically** — even if an error occurs inside the block.

```python
# Without with — risky:
f    = open('file.txt')
data = f.read()    # if this raises an error, f.close() is NEVER called
f.close()

# With the with statement — safe:
with open('file.txt') as f:
    data = f.read()    # file is always closed when the block exits
```

| Mode | String | Behaviour |
|------|--------|-----------|
| Read | `'r'` | Open existing file. `FileNotFoundError` if missing. |
| Write | `'w'` | Create file. **Overwrites** if already exists. |
| Append | `'a'` | Add to the end. Creates file if missing. |

> Always add `encoding='utf-8'` to avoid silent text corruption on Windows.


### Reading a file — `load_system_prompt()`


In [ ]:
def load_system_prompt(filepath='sample_prompt.txt'):
    """
    Load a system prompt from a .txt file.
    Keeping prompts in files means the team can edit them without touching Python.
    """
    # 'r' = read mode  |  encoding='utf-8' = always specify
    # with block closes the file automatically — even if f.read() raises an error
    with open(filepath, 'r', encoding='utf-8') as f:
        content = f.read()   # loads the entire file as one string

    # .strip() removes the trailing newline that text editors always add
    return content.strip()


In [ ]:
# Write a prompt to a file — simulates prompts/system_prompt.txt
with open('sample_prompt.txt', 'w', encoding='utf-8') as f:
    f.write('## Role\n')
    f.write('You are a customer support assistant for ShopSmart.\n')
    f.write('\n')
    f.write('## Task\n')
    f.write('Answer using ONLY the information provided.')

In [ ]:
# Now call the function — reads the file back
prompt = load_system_prompt('sample_prompt.txt')
print(f'Loaded {len(prompt)} characters:')
print(prompt)

### Appending to a file — `save_llm_response_log()` — uses `json.dumps()`


In [ ]:
def save_llm_response_log(query, response, log_file='llm_calls.log'):
    """
    Append one LLM call to a JSONL log file.
    JSONL = JSON Lines: one JSON object per line.
    """
    # 'a' = append mode:
    #   file exists  → adds to the END, does not overwrite previous entries
    #   file missing → creates it automatically
    with open(log_file, 'a', encoding='utf-8') as f:
        # json.dumps() converts a Python dict → JSON string
        entry = json.dumps({'query': query, 'response': response})
        f.write(entry + '\n')   # one entry per line = JSONL format


In [ ]:
# Call the function twice — 'a' mode appends each call to the same file
# TRACE:
# Call 1 → file missing → 'a' creates llm_calls.log → writes line 1
# Call 2 → file exists  → 'a' appends              → writes line 2 (line 1 untouched)

In [ ]:
save_llm_response_log('Where is my order?', 'Your order ORD-3042 is In Transit.')

In [ ]:
save_llm_response_log('Can I return this?', 'Returns accepted within 30 days.')

In [ ]:
# Read back every line — same filename the function wrote to
# TRACE:
# line 0 → '{"query": "Where is my order?", "response": "..."}\n'
#           .strip() removes \n  →  json.loads() converts string → dict
# line 1 → second entry, same process
print('Log contents:')
with open('llm_calls.log', 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        record = json.loads(line.strip())
        print(f'  [{i+1}] Q: {record["query"]}')
        print(f'       A: {record["response"]}')

---


### Optional extension: parsing JSON returned by an LLM

Sometimes an LLM wraps JSON in Markdown code fences such as ```json. The helper below removes those fences before parsing. Learn the basic `json.loads()` example first; return to this helper when you're comfortable.

In [ ]:
def parse_llm_json_response(raw_response):
    """
    Parse a JSON string from the LLM into a Python dict.
    Also handles responses wrapped in markdown fences:
        ```json
        {"key": "value"}
        ```
    """
    # Step 1: remove leading/trailing whitespace
    cleaned = raw_response.strip()

    # Step 2: strip markdown fences if present
    # LLMs often add ```json ... ``` even when told not to
    if cleaned.startswith('```'):
        # split('\n', 1) → split on first newline only
        # [-1] takes everything after the opening fence line
        cleaned = cleaned.split('\n', 1)[-1]
        # rsplit('```', 1) → split on last ``` only
        # [0] takes everything before the closing fence
        cleaned = cleaned.rsplit('```', 1)[0].strip()

    # Step 3: convert JSON string → Python dict
    return json.loads(cleaned)


In [ ]:
# ── Demo 1: plain JSON string ────────────────────────────────
raw = '{"category": "TRACK_ORDER", "confidence": "high"}'

print('Before parse — type:', type(raw).__name__)   # str
parsed = parse_llm_json_response(raw)
print('After parse  — type:', type(parsed).__name__) # dict
print('category  :', parsed['category'])
print('confidence:', parsed['confidence'])


In [ ]:
# ── Demo 2: LLM wrapped response in markdown fences ──────────
# This is very common — LLMs add ```json even when told not to
fenced = '```json\n{"intent": "CANCEL", "urgency": "high"}\n```'

print('Raw response from LLM:')
print(fenced)
print()

# TRACE:
# cleaned = fenced.strip()
# startswith('```') → True
# split('\n',1)[-1]  → '{"intent": "CANCEL", "urgency": "high"}\n```'
# rsplit('```',1)[0]  → '{"intent": "CANCEL", "urgency": "high"}\n'
# .strip()            → '{"intent": "CANCEL", "urgency": "high"}'
# json.loads()        → {'intent': 'CANCEL', 'urgency': 'high'}

parsed2 = parse_llm_json_response(fenced)
print('After parse:', parsed2)
print('intent    :', parsed2['intent'])


---


## 4 — Reading CSV Data with `csv.DictReader`

`csv.DictReader` reads the first row as column headers and maps every subsequent row to those headers.  
Each row becomes a dict — exactly the same shape as a database row in Python.

> **Important:** every CSV value is a **string**.  
> Always convert: `int(row['customer_id'])`, `float(row['price'])`.


### Reading customer rows from CSV


In [ ]:
import csv
# sample_csv simulates what customers.csv looks like on disk
# io.StringIO wraps a string so csv.DictReader treats it like an open file
with open('employee.csv', 'r', encoding='utf-8') as f:
    # csv.DictReader reads row 1 as headers, maps each subsequent row to them
    # ITERATION TRACE:
    # Row 1 → {'customer_id':'1001', 'name':'Prudhvi Akella', 'city':'Rajahmundry', ...}
    # Row 2 → {'customer_id':'1002', 'name':'Ravi Kumar',     'city':'Hyderabad', ...}
    # Row 3 → {'customer_id':'1003', 'name':'Ananya Sharma',  'city':'Bangalore', ...}
    # EOF   → loop ends

    reader    = csv.DictReader(f)
    customers = list(reader)
    print(customers)

In [ ]:
for c in customers:
    print(f"  {c['customer_id']}  {c['name']:20s}  {c['city']}, {c['state']}")

print()
print('--- CSV type trap ---')
print('customer_id type    :', type(customers[0]['customer_id']).__name__, '← str!')
print("String compare trap : '1001' > '200' =", '1001' > '200', '← WRONG (alphabetic)')
print('Correct int compare : 1001  > 200  =',   1001  > 200)
print('Fix                 :', int(customers[0]['customer_id']))


### Optional extension: other CSV delimiters

Most CSV files use commas. Some use semicolons, tabs, or pipes. Learn the standard comma-separated example first, then use these variations when you encounter them.

In [ ]:
# ── Semicolon-delimited (common in European exports) ────────
semicolon_csv = """customer_id;name;city
1001;Prudhvi Akella;Rajahmundry
1002;Ravi Kumar;Hyderabad
"""

reader = csv.DictReader(io.StringIO(semicolon_csv), delimiter=';')
for row in reader:
    print(row)


In [ ]:
# ── Tab-separated / TSV ─────────────────────────────────────
tsv_data = 'customer_id\tname\tcity\n1001\tPrudhvi Akella\tRajahmundry\n1002\tRavi Kumar\tHyderabad'

reader = csv.DictReader(io.StringIO(tsv_data), delimiter='\t')
for row in reader:
    print(row)


In [ ]:
# ── Pipe-separated ──────────────────────────────────────────
pipe_csv = """customer_id|name|city
1001|Prudhvi Akella|Rajahmundry
1002|Ravi Kumar|Hyderabad
"""

reader = csv.DictReader(io.StringIO(pipe_csv), delimiter='|')
for row in reader:
    print(row)


### Optional extension: CSV without a header row

If a CSV file has no header row, use `fieldnames` to supply column names manually.

In [ ]:
# No header row — supply column names with fieldnames=
no_header_csv = """1001,Prudhvi Akella,Rajahmundry
1002,Ravi Kumar,Hyderabad
"""

reader = csv.DictReader(
    io.StringIO(no_header_csv),
    fieldnames=['customer_id', 'name', 'city']   # supply headers manually
)
for row in reader:
    print(row)


### Optional extension: detect a CSV delimiter

`csv.Sniffer` can guess the delimiter from a sample. This is useful, but not required for the first lesson.

In [ ]:
# csv.Sniffer — auto-detect delimiter
sample = """customer_id,name,email,city,state
1001,Prudhvi Akella,"prudhvi@example.com,prudhvi1@example2.com",Rajahmundry,AP
1002,Ravi Kumar,ravi@example.com,Hyderabad,TS
1003,Ananya Sharma,ananya@example.com,Bangalore,KA
"""

dialect  = csv.Sniffer().sniff(sample)   # sniff() reads a sample and guesses
print(type(dialect))
print('Detected delimiter:', repr(dialect.delimiter))
print("Quote character:", repr(dialect.quotechar))

In [ ]:
# Use the detected dialect in DictReader
reader = csv.DictReader(io.StringIO(sample), dialect=dialect)
for row in reader:
    print(row)

---


## 5 — Putting it together: a small AI-style workflow

This is the **application exercise**. It combines the skills from earlier sections. Run it after you understand file handling, JSON, and CSV.

### Step 1 — Load the system prompt from a file


In [ ]:
system_prompt = load_system_prompt('sample_prompt.txt')

print(f'Loaded {len(system_prompt)} chars')
print(system_prompt)


### Step 2 — Read customer data


In [ ]:
# In production this row comes from csv.DictReader (Section 4)
csv_row       = {'customer_id': '1001', 'name': 'Prudhvi Akella', 'city': 'Rajahmundry'}
customer_id   = int(csv_row['customer_id'])   # CSV always gives str → convert
customer_name = csv_row['name']

print(f'Customer: {customer_name}  id={customer_id}  type={type(customer_id).__name__}')


### Step 3 — Build the user message with an f-string


In [ ]:
user_message = f'Customer: {customer_name}\nQuery: Where is my order #3042?'

print(user_message)


### Step 4 — Receive LLM response (simulated with fences)


In [ ]:
# In production this comes back from openai.chat.completions.create()
mock_llm_output = '```json\n{"category": "TRACK_ORDER", "reply": "Your order ORD-3042 is In Transit.", "confidence": "high"}\n```'

print('Raw LLM output:')
print(mock_llm_output)


### Step 5 — Parse the response


In [ ]:
parsed     = parse_llm_json_response(mock_llm_output)
category   = parsed.get('category',   'UNKNOWN')
reply      = parsed.get('reply',       'No reply')
confidence = parsed.get('confidence',  'unknown')

print('Parsed dict :', parsed)
print('category    :', category)
print('reply       :', reply)
print('confidence  :', confidence)


### Step 6 — Log the call


In [ ]:
save_llm_response_log(user_message, reply)

# Read back to confirm it was written
with open('llm_calls.log', 'r', encoding='utf-8') as f:
    lines = f.readlines()

print(f'llm_calls.log now has {len(lines)} entries')
print('Last entry:', lines[-1].strip())


---


## Summary — Day 04

| Concept | Remember |
|---|---|
| JSON | A text format for structured data |
| `json.dumps(data)` | Python object → JSON string |
| `json.loads(text)` | JSON string → Python object |
| `json.dump(data, file)` | Write Python data to a JSON file |
| `json.load(file)` | Read a JSON file into Python data |
| `with open(...) as file` | Closes the file automatically |
| Mode `'r'` | Read an existing file |
| Mode `'w'` | Write; overwrites an existing file |
| Mode `'a'` | Append to the end of a file |
| `csv.DictReader` | Reads each CSV row as a dictionary |
| CSV values | Usually strings; convert numbers when needed |

**Optional extensions:** parsing Markdown-fenced LLM responses, unusual CSV delimiters, missing CSV headers, and `csv.Sniffer`.

**Next:** Day 05 — Exception Handling + Retry + Logging.